# Tayseer: SAR 40M for eight regions below target
SDAIA Academy SDA-DSC-112. All data are synthetic.
The final presentation and [shared Tableau reference](https://public.tableau.com/views/Dvis_17912834975050/TayseerDigitalAdoption) use user-weighted adoption. This is an index; users may overlap across records. The published reference shows December 2025; the monthly trend below is computed from the full CSV.
Run locally with the repository requirements. In Colab upload the supplied CSV if prompted.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
paths=[Path('data/tayseer_services_synthetic.csv'),Path('../data/tayseer_services_synthetic.csv'),Path('tayseer_services_synthetic.csv')]
source=next((p for p in paths if p.exists()),None)
if source is None:
    from google.colab import files
    uploaded=files.upload()
    source=Path(next(k for k in uploaded if k.endswith('.csv')))
df=pd.read_csv(source,parse_dates=['month'])
print('Source:',source,'Shape:',df.shape)

Source: data\tayseer_services_synthetic.csv Shape: (24960, 12)


## 1. Source quality

In [2]:
keys=['month','region','service_category','channel']
assert df.shape==(24960,12)
assert not df.isna().any().any() and not df.duplicated(keys).any()
assert [df[k].nunique() for k in keys]==[48,13,8,5]
assert df.digital_adoption_pct.between(0,100).all()
assert (df.unique_users>0).all()
latest=df.month.max()
current=df.loc[df.month.eq(latest)].copy()
print('Full source grid; no missing cells or duplicate keys.')
print('Latest month:',latest.strftime('%Y-%m'))

Full source grid; no missing cells or duplicate keys.
Latest month: 2025-12


## 2. Weighted national trend
SUM(adoption percentage × users) / SUM(users). Percentages already use a 0–100 scale.

In [3]:
def weighted(rs):
    return float((rs.digital_adoption_pct*rs.unique_users).sum()/rs.unique_users.sum())
trend=pd.DataFrame([{'month':m,'adoption_pct':weighted(rs)} for m,rs in df.groupby('month')])
national=weighted(current)
first=float(trend.adoption_pct.iloc[0])
print(f'January 2022: {first:.6f}%; December 2025: {national:.6f}%')
print(f'Growth: {national-first:.6f} pp; above target: {national-65:.6f} pp')
fig=px.line(trend,x='month',y='adoption_pct',title='National weighted adoption, Jan 2022 to Dec 2025')
fig.update_yaxes(title='Adoption (%)',range=[0,100])
fig.add_hline(y=65,line_dash='dash',line_color='#C25D10',annotation_text='65% target')
fig.update_traces(line_color='#087E87')
fig.show()

January 2022: 54.142142%; December 2025: 66.210761%
Growth: 12.068619 pp; above target: 1.210761 pp


Interactive Plotly chart. Trust the notebook to display.

## 3. Regional complication
Every below-target region is emphasized, including Al-Baha.

In [4]:
regional=pd.DataFrame([{'region':r,'adoption_pct':weighted(rs)} for r,rs in current.groupby('region')])
regional['gap_pp']=np.maximum(65-regional.adoption_pct,0)
regional['target_status']=np.where(regional.gap_pp.gt(0),'Below Target','On/Above Target')
regional=regional.sort_values(['gap_pp','region'],ascending=[False,True]).reset_index(drop=True)
funded=regional.loc[regional.gap_pp.gt(0)].copy().reset_index(drop=True)
assert len(funded)==8
print('Below target:',len(funded),'of',len(regional))
print(regional.to_string(index=False))
bars=regional.sort_values('adoption_pct')
fig=px.bar(bars,x='adoption_pct',y='region',orientation='h',color='target_status',
           color_discrete_map={'Below Target':'#087E87','On/Above Target':'#D8E6E7'},
           title='Eight regions below target, December 2025')
fig.update_xaxes(range=[0,100],title='Adoption (%)')
fig.update_yaxes(categoryorder='array',categoryarray=bars.region.tolist(),autorange='reversed')
fig.add_vline(x=65,line_color='#C25D10',annotation_text='65% target')
fig.show()

Below target: 8 of 13
          region  adoption_pct   gap_pp   target_status
          Najran     60.863719 4.136281    Below Target
Northern Borders     62.700224 2.299776    Below Target
         Al-Baha     62.781318 2.218682    Below Target
           Jazan     63.028712 1.971288    Below Target
            Asir     64.357850 0.642150    Below Target
           Tabuk     64.484609 0.515391    Below Target
            Hail     64.742108 0.257892    Below Target
         Al-Jouf     64.821701 0.178299    Below Target
Eastern Province     68.702701 0.000000 On/Above Target
         Madinah     68.232440 0.000000 On/Above Target
          Makkah     70.697932 0.000000 On/Above Target
          Qassim     66.105987 0.000000 On/Above Target
          Riyadh     70.750772 0.000000 On/Above Target


Interactive Plotly chart. Trust the notebook to display.

## 4. Four widest gaps
The share is of summed positive gaps, not population or transactions. Compare equal 5M per laggard, concentrated 10M in each of four, and gap-based coverage of all eight.

In [5]:
total_gap=float(funded.gap_pp.sum())
share=float(funded.head(4).gap_pp.sum()/total_gap*100)
print('Four widest:',funded.head(4).region.tolist())
print(f'Four-region combined gap share: {share:.6f}%')
fig=px.bar(funded,x='gap_pp',y='region',orientation='h',title='Positive regional gaps to 65%')
fig.update_xaxes(title='Gap (percentage points)',range=[0,5])
fig.update_yaxes(categoryorder='array',categoryarray=funded.region.tolist(),autorange='reversed')
fig.show()

Four widest: ['Najran', 'Northern Borders', 'Al-Baha', 'Jazan']
Four-region combined gap share: 86.957753%


Interactive Plotly chart. Trust the notebook to display.

## 5. Exact allocation
Distribute the full SAR 40M proportional to gaps. Round to half a million using largest remainders, preserving the total. This is a proposal, not a local cost estimate.

In [6]:
ideal_units=(40*funded.gap_pp/total_gap/0.5).to_numpy()
units=np.floor(ideal_units).astype(int)
remaining=80-int(units.sum())
order=np.argsort(-(ideal_units-units),kind='stable')
units[order[:remaining]]+=1
funded['allocation_sar_m']=units*0.5
assert funded.allocation_sar_m.tolist()==[13.5,7.5,7.5,6.5,2,1.5,1,0.5]
assert funded.allocation_sar_m.sum()==40
assert funded.head(4).allocation_sar_m.sum()==35
assert funded.tail(4).allocation_sar_m.sum()==5
print(funded[['region','gap_pp','allocation_sar_m']].to_string(index=False))
print('First four: 35M; other four: 5M; all regions: 40M.')

          region   gap_pp  allocation_sar_m
          Najran 4.136281              13.5
Northern Borders 2.299776               7.5
         Al-Baha 2.218682               7.5
           Jazan 1.971288               6.5
            Asir 0.642150               2.0
           Tabuk 0.515391               1.5
            Hail 0.257892               1.0
         Al-Jouf 0.178299               0.5
First four: 35M; other four: 5M; all regions: 40M.


## 6. Reconciliation and implementation
Approve and appoint a lead. Validate local costs and service plans within 30 days before spending; review adoption, spending and service quality after six months. These are proposed checkpoints. The source cannot establish a causal numeric response to spending. Aim to move each lagging region toward 65%.

In [7]:
import json,hashlib
summary_path=next((p for p in [Path('analysis/summary.json'),Path('summary.json')] if p.exists()),None)
if summary_path:
    expected=json.loads(summary_path.read_text(encoding='utf-8'))
    assert hashlib.sha256(source.read_bytes()).hexdigest()==expected['source_sha256']
    assert abs(national-expected['national_latest_pct'])<1e-10
    assert abs(first-expected['national_first_pct'])<1e-10
    assert abs(share-expected['priority_regions_gap_share_pct'])<1e-10
    assert funded.region.tolist()==expected['selected_regions']
    assert funded.allocation_sar_m.tolist()==[r['allocation_sar_m'] for r in expected['selected_allocations']]
    print('Independent notebook matches source, summary, gap share and all allocations.')
else:
    print('Standalone source calculation complete; packaged summary unavailable in this environment.')
print('Synthetic data; overlapping user weights; costs and causal effects require local evidence.')

Independent notebook matches source, summary, gap share and all allocations.
Synthetic data; overlapping user weights; costs and causal effects require local evidence.


## Presentation
Seven English slides with Arabic presenter notes. State the request within 30 seconds, repeat at closing, and rehearse within seven minutes. Use Tableau for supporting evidence and Q&A after verifying desktop opening.